# Cosmetics Retail Sales Analysis

> **Project context:** This notebook analyzes the cosmetics sales dataset used in the Smart Cosmetics Retail Management System Business Analysis case study. The analysis is intended for educational and portfolio purposes.

## Objectives

This analysis aims to:
- understand the structure and quality of the sales data;
- measure sales performance by country, salesperson, product, and month;
- identify important sales patterns and differences;
- translate quantitative findings into business insights;
- provide evidence for the business problems and requirements defined in the Business Analysis section.


## 1. Import Libraries and Load Data

Pandas is used for data manipulation and analysis, while Matplotlib is used for visualization.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

file_path = "../../cosmetics_sales_data.xlsx"
df = pd.read_excel(file_path, sheet_name="cosmetics_sales_data")

df.head()


In [ ]:
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")
print("Columns:", df.columns.tolist())


## 2. Dataset Overview

The main worksheet contains transaction-level sales records for salespeople, countries, products, dates, sales amounts, boxes shipped, and unit prices.


In [ ]:
df.info()


In [ ]:
print("Salespeople:", df["Sales Person"].nunique())
print("Countries:", df["Country"].nunique())
print("Products:", df["Product"].nunique())
print("Date range:", df["Date"].min().date(), "to", df["Date"].max().date())
print("Total sales: $", round(df["Amount ($)"].sum(), 2))


## 3. Data Quality Checks

Before analysis, missing values, duplicate rows, and the relationship between sales amount, quantity shipped, and unit price are checked.


In [ ]:
print("Missing values:")
print(df.isnull().sum())
print("\nExact duplicate rows:", df.duplicated().sum())


In [ ]:
df["Calculated Amount"] = df["Boxes Shipped"] * df["Unit Price"]
df["Amount Difference"] = (df["Amount ($)"] - df["Calculated Amount"]).abs()

print("Maximum calculation difference:", df["Amount Difference"].max())
print("Rows with difference > $0.01:", (df["Amount Difference"] > 0.01).sum())


### Data Quality Interpretation

The main transaction table contains no missing values or exact duplicate rows. The recorded `Amount ($)` is also consistent with `Boxes Shipped × Unit Price` within normal floating-point precision. This provides a suitable baseline for descriptive analysis.

This does not prove that a future production system would always contain clean data; it supports the later requirement for preventive data-validation controls.


## 4. Overall Sales Performance


In [ ]:
total_sales = df["Amount ($)"].sum()
total_transactions = len(df)
total_boxes = df["Boxes Shipped"].sum()
average_transaction = df["Amount ($)"].mean()

summary = pd.Series({
    "Total Sales ($)": total_sales,
    "Transactions": total_transactions,
    "Boxes Shipped": total_boxes,
    "Average Transaction ($)": average_transaction
})
summary


## 5. Sales by Country


In [ ]:
country_sales = (
    df.groupby("Country")["Amount ($)"]
    .agg(["sum", "count", "mean"])
    .sort_values("sum", ascending=False)
    .rename(columns={
        "sum": "Total Sales ($)",
        "count": "Transactions",
        "mean": "Average Transaction ($)"
    })
)
country_sales["Sales Share (%)"] = country_sales["Total Sales ($)"] / total_sales * 100
country_sales


In [ ]:
plt.figure(figsize=(9, 5))
plt.bar(country_sales.index, country_sales["Total Sales ($)"])
plt.title("Total Sales by Country")
plt.xlabel("Country")
plt.ylabel("Sales ($)")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()


### Country Insight

The United States is the highest-sales market, followed by New Zealand and Australia. India has the lowest total sales among the six countries. This supports the need for regional sales visibility and comparison.

The analysis shows performance differences but does not explain their causes.


## 6. Sales by Product


In [ ]:
product_sales = (
    df.groupby("Product")["Amount ($)"]
    .agg(["sum", "count", "mean"])
    .sort_values("sum", ascending=False)
    .rename(columns={
        "sum": "Total Sales ($)",
        "count": "Transactions",
        "mean": "Average Transaction ($)"
    })
)
product_sales["Sales Share (%)"] = product_sales["Total Sales ($)"] / total_sales * 100
product_sales


In [ ]:
plt.figure(figsize=(10, 6))
plt.barh(product_sales.index[::-1], product_sales["Total Sales ($)"].iloc[::-1])
plt.title("Sales by Product")
plt.xlabel("Sales ($)")
plt.ylabel("Product")
plt.tight_layout()
plt.show()


### Product Insight

Tea Tree Moisturizer is the highest-revenue product, while Charcoal Face Wash is the lowest-revenue product in the analyzed period. The variation supports product-level performance monitoring.

Revenue differences should not be interpreted as profitability differences because product cost and margin data are not available.


## 7. Sales by Salesperson


In [ ]:
salesperson_sales = (
    df.groupby("Sales Person")["Amount ($)"]
    .agg(["sum", "count", "mean"])
    .sort_values("sum", ascending=False)
    .rename(columns={
        "sum": "Total Sales ($)",
        "count": "Transactions",
        "mean": "Average Transaction ($)"
    })
)
salesperson_sales


In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(salesperson_sales.index, salesperson_sales["Total Sales ($)"])
plt.title("Sales by Salesperson")
plt.xlabel("Salesperson")
plt.ylabel("Sales ($)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()


### Salesperson Insight

Olivia D'Souza records the highest total sales, while Mason Kapoor records the lowest among the listed salespeople. Revenue differences may be influenced by transaction volume, market allocation, product mix, or other factors, so the dataset does not establish the cause of the variation.


## 8. Monthly Sales Trend


In [ ]:
df["Month"] = df["Date"].dt.to_period("M")

monthly_sales = (
    df.groupby("Month")["Amount ($)"]
    .sum()
    .reset_index()
)
monthly_sales["Month"] = monthly_sales["Month"].astype(str)
monthly_sales


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(monthly_sales["Month"], monthly_sales["Amount ($)"], marker="o")
plt.title("Monthly Sales Trend")
plt.xlabel("Month")
plt.ylabel("Sales ($)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


### Monthly Trend Insight

Sales increased substantially from February (approximately $214,025) to March (approximately $484,102), followed by lower sales toward August (approximately $275,299). This supports the need for sales trend monitoring.

The dataset does not explain why the changes occurred. Possible causes such as promotions, seasonality, pricing, or demand would require additional data or stakeholder investigation.


## 9. Sales Concentration


In [ ]:
top_3_countries_share = country_sales["Total Sales ($)"].head(3).sum() / total_sales * 100
top_5_products_share = product_sales["Total Sales ($)"].head(5).sum() / total_sales * 100

print(f"Top 3 countries' sales share: {top_3_countries_share:.2f}%")
print(f"Top 5 products' sales share: {top_5_products_share:.2f}%")


## 10. Key Findings

1. The dataset contains **374 transactions** and approximately **$2.91 million** in recorded sales.
2. The **United States** is the highest-sales country at approximately **$628,488**.
3. **New Zealand** and **Australia** are the next two highest-sales markets.
4. **Tea Tree Moisturizer** is the highest-revenue product at approximately **$260,905**.
5. **Charcoal Face Wash** is the lowest-revenue product at approximately **$102,733**.
6. **Olivia D'Souza** has the highest total salesperson revenue at approximately **$387,406**.
7. Monthly sales fluctuate substantially, with a major increase from February to March and lower sales toward August.
8. The transaction data passes the basic quality checks performed in this notebook.


## 11. Business Implications

| Data Finding | Business Implication | Potential System Capability |
|---|---|---|
| Different sales levels by country | Geographic performance needs monitoring | Regional sales dashboard |
| Different salesperson revenue | Management needs performance visibility | Salesperson performance reporting |
| Different product revenue | Product performance needs comparison | Product analytics |
| Large monthly fluctuations | Sales trends should be monitored | Time-based dashboard |
| Analysis is required to produce management insights | Reporting can benefit from automation | Automated reports and KPIs |
| Dataset lacks stock-on-hand information | Inventory is an information gap | Inventory management module |
| Dataset is clean but future data may grow | Preventive validation is useful | Data validation controls |

> **Important:** The dataset does not prove actual stockouts, customer-retention problems, profitability problems, or supplier problems. Those areas require additional data or stakeholder validation.


## 12. Dataset Limitations

The dataset does not include:
- customer profiles or customer IDs;
- inventory stock-on-hand quantities;
- reorder points or stockout records;
- supplier information;
- payment status;
- delivery status;
- product cost;
- profit or profit margin;
- marketing campaign information.

Consequently, this notebook focuses on descriptive sales performance rather than profitability, customer behavior, inventory optimization, or causal analysis.


## 13. Connection to Business Analysis

The analytical chain is:

**Raw Sales Data → Data Cleaning & Validation → Descriptive Analysis → Business Findings → Business Implications → Pain Points / Opportunities → Requirements**

Example:

**Finding:** Monthly sales vary substantially.

**Business implication:** Management needs faster visibility into sales trends.

**Business need:** Sales performance should be monitored through centralized reporting.

**Potential requirement:** The system should allow authorized users to view sales trends by date range.

This approach connects proposed system requirements to quantitative business evidence rather than selecting features only because they are technically interesting.


## 14. Conclusion

The cosmetics sales analysis demonstrates meaningful differences in sales performance across countries, products, salespeople, and months. The dataset is sufficiently consistent for descriptive analysis and provides a quantitative foundation for the Business Analysis case study.

The main opportunity is to transform transaction-level data into accessible management information through centralized reporting and analytics. The dataset limitations also demonstrate why a complete retail management system would require additional information about customers, orders, inventory, payments, and fulfillment.

The findings from this notebook will support the **Business Requirements**, **Functional Requirements**, **Process Modeling**, **System Analysis**, and **Database Design** stages of the project.
